# RegFM single-query demo

Predict one gene expression value from:
- a DNA sequence string (`"AGCT..."`)
- a length-2103 TF/CR expression vector (same order as the TF vocab)

Download pretrained checkpoints from Hugging Face:
**https://huggingface.co/Deku21/RegFM**

```bash
hf download Deku21/RegFM --local-dir ./RegFM_weights
```

Then set `MODEL_PATH` (and tokenizer / config paths) below. Run from the repo root.
Demo inputs are for illustration.


## Setup

In [12]:
import os
import sys
from argparse import Namespace
from pathlib import Path

import numpy as np
import torch
from transformers import BertConfig, DNATokenizer

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# Paths after downloading from https://huggingface.co/Deku21/RegFM
MODEL_PATH = "/path/to/RegFM_weights"
DNA_TOKENIZER = "/path/to/dna_tokenizer.json"
TRANS_TOKENIZER = "/path/to/tf_vocab.txt"
EXP_TOKENIZER = "/path/to/exp_vocab.txt"
EXP_CONFIG = "/path/to/exp_config.json"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("model :", MODEL_PATH)


device: cuda


In [23]:
from utils import build_dna_tokenizer, load_finetuned_checkpoint

def tfcr_order_from_vocab(vocab_path):
    with open(vocab_path) as f:
        tokens = [line.strip().split()[0] for line in f if line.strip()]
    # vocab[0:5] are special tokens; vocab[5:] are the 2103 TF/CRs
    names = tokens[5:]
    assert len(names) == 2103, f"expected 2103 TF/CRs, got {len(names)}"
    return names

def pad(ids, max_len, pad_id):
    ids = list(ids)[:max_len]
    mask = [1] * len(ids) + [0] * (max_len - len(ids))
    return ids + [pad_id] * (max_len - len(ids)), mask

tran_max_len = 2112
cis_max_len = 71680

dna_tokenizer = build_dna_tokenizer(Namespace(dna_tokenizer_name=DNA_TOKENIZER))
tf_tokenizer = DNATokenizer.from_pretrained(TRANS_TOKENIZER)
exp_tokenizer = DNATokenizer.from_pretrained(EXP_TOKENIZER)
tfcr_order = tfcr_order_from_vocab(TRANS_TOKENIZER)

config = BertConfig.from_pretrained(EXP_CONFIG, num_labels=1, finetuning_task="genepred")
config.hidden_dropout_prob = 0.1
config.attention_probs_dropout_prob = 0.1
model = load_finetuned_checkpoint(MODEL_PATH, device, config=config)
model.to(device).eval()
print(type(model).__name__, "on", device, "| #TF/CR =", len(tfcr_order))

Calling DNATokenizer.from_pretrained() with the path to a single file or url is deprecated
Calling DNATokenizer.from_pretrained() with the path to a single file or url is deprecated


<class 'transformers.tokenization_dna.DNATokenizer'>
<class 'transformers.tokenization_dna.DNATokenizer'>
RegFM on cuda | #TF/CR = 2103


## Predict

`predict_one(dna_seq, trans_values)` takes a DNA string and a length-2103 vector, returns one float.

In [24]:
@torch.no_grad()
def predict_one(dna_seq, trans_values):
    dna_seq = dna_seq.strip().upper()
    trans_values = np.asarray(trans_values).reshape(-1)
    assert len(trans_values) == len(tfcr_order)

    tf_ids, tf_mask = pad(
        tf_tokenizer.encode(" ".join(tfcr_order), add_special_tokens=True),
        tran_max_len, tf_tokenizer.pad_token_id or 0,
    )
    trans_ids, _ = pad(
        exp_tokenizer.encode(" ".join(map(str, map(int, trans_values))), add_special_tokens=True),
        tran_max_len, exp_tokenizer.pad_token_id or 0,
    )
    dna_ids, dna_mask = pad(
        dna_tokenizer.encode(dna_seq, add_special_tokens=True),
        cis_max_len, dna_tokenizer.pad_token_id or 0,
    )

    out = model(
        input_ids=torch.tensor([tf_ids], device=device),
        attention_mask=torch.tensor([tf_mask], device=device),
        trans_ids=torch.tensor([trans_ids], device=device),
        dna_ids=torch.tensor([dna_ids], device=device),
        dna_attention_mask=torch.tensor([dna_mask], device=device),
    )
    return float(out[0].view(-1)[0].cpu())

In [25]:
np.random.seed(0)

# Two inputs (demo values):
# 1) DNA sequence string
dna = "".join(np.random.choice(list("ACGT"), size=cis_max_len))
# 2) TF/CR expression vector, length = len(tfcr_order) (== 2103)
trans = np.random.randint(0, 256, size=len(tfcr_order))

# One output: predicted expression
y = predict_one(dna, trans)
print(dna[:48] + "...")
print("trans shape:", trans.shape)
print("prediction:", y)


ATCATTTTCTCGATGAAAGCGTTGACCCCACATATCGTTAGTACTCTT...
trans shape: (2103,)
prediction: 2.0560576915740967
